In [8]:
import importlib
import utils.functions as functions 
import utils.pre_processing as pre_processing
importlib.reload(pre_processing)
from utils.pre_processing import *
importlib.reload(functions)
from utils.functions import *

#org
import shutil 
import torch
import numpy as np
import warnings
import os, re, glob
from scipy import interpolate
import pandas as pd
from copy import deepcopy

#images
import matplotlib.gridspec as gridspec
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import tifffile as tiff
from matplotlib.animation import FuncAnimation, PillowWriter

#scientific libraries
from applefy.utils import flux_ratio2mag, mag2flux_ratio
from applefy.utils.photometry import AperturePhotometryMode
from applefy.detections.contrast import Contrast
from applefy.utils.photometry import AperturePhotometryMode
from applefy.statistics import TTest, gaussian_sigma_2_fpf, LaplaceBootstrapTest



from pathlib import Path
root_dir = Path(".")
print(root_dir)

.


# Test display settings effect

I took 7 images where I changed the display mode (normal, logscale), and the contrast, and I want to check if it had any effect on the frames taken.

In [ ]:
folder      = '/home/aosimul/noah/data/ghost_images/phase_screens_SLM/test_display/'
#ros = [0.50, 0.70, 1.00]
test = np.concatenate([
    tiff.imread(
        sorted(glob.glob(f"{folder}*.tif"))
    )
    #for windspeed in ros
])
test  = test.astype(np.float32) #/ 65535

In [ ]:
test_croped = crop_stack(test)
print(test_croped.shape)

In [ ]:
plt.figure(figsize = (10,4))
plt.subplot(1,2,1)
plt.imshow(np.mean(test_croped, axis = 0))
plt.colorbar(label = 'Counts')
plt.title('Mean PSF')

plt.subplot(1,2,2)
plt.imshow(np.std(test_croped, axis = 0), vmin =0)
plt.colorbar(label = 'Counts')
plt.title('STD PSF')

In [ ]:
test_croped  = test_croped.astype(np.float32) / 65535

plt.figure(figsize = (10,4))
plt.subplot(1,2,1)
plt.imshow(np.mean(test_croped, axis = 0))
plt.colorbar(label = 'Fraction of Saturation')
plt.title('Mean PSF')

plt.subplot(1,2,2)
plt.imshow(np.std(test_croped, axis = 0), vmin =0)
plt.colorbar(label = 'Fraction of Saturation')
plt.title('STD PSF')

plt.suptitle('Numerical error')

# Pre-Processing of one stack

- Create master dark frame: median of 3500 dark frames of 25ms, in counts
- Center
- Crop
- Subtract df
- divide by 65535


In [ ]:
radius = 55

In [ ]:
folder      = '/home/aosimul/noah/data/ghost_images/phase_screens_SLM/7_22/darks_2026-07-22T10-28-49.8'
df = tiff.imread(glob.glob(f"{folder}*.tif"))
#df  = df.astype(np.float32) #/ 65535
df = crop_stack(df, radius = radius)
master_df = np.median(df, axis = 0)
print(master_df.shape)

In [ ]:
folder      = '/home/aosimul/noah/data/ghost_images/phase_screens_SLM/7_22/psf'
sci_img = tiff.imread(glob.glob(f"{folder}*.tif"), key=range(3500))
sci_img = crop_stack(sci_img,  radius = radius)
sci_img_processed = (sci_img - master_df) 
sci_img_processed  = sci_img_processed.astype(np.float32) / 65535
#np.save(f'/home/aosimul/noah/data/ghost_images/25ms/coro_int_0.70ro_12.50ws.npy', sci_img_processed.astype(np.float32))

# folder      = '/home/aosimul/noah/data/ghost_images/phase_screens_SLM/7_17/non_coro_int_0.70ro_12.50ws'
# sci_img = tiff.imread(glob.glob(f"{folder}*.tif"), key=range(3500))
# sci_img = crop_stack(sci_img,  radius = radius)
# sci_img_processed = (sci_img - master_df) 
# sci_img_processed  = sci_img_processed.astype(np.float32) / 65535
# #np.save(f'/home/aosimul/noah/data/ghost_images/25ms/non_coro_int_0.70ro_12.50ws.npy', sci_img_processed.astype(np.float32))


print(sci_img_processed.shape)

In [ ]:
# non_coro_int  = np.load('/home/aosimul/noah/data/ghost_images/25ms/non_coro_int_0.70ro_12.50ws.npy')
# non_coro_int = np.load('/home/aosimul/noah/data/ghost_images/25ms/coro_int_0.70ro_12.50ws.npy')
# print(non_coro_int.shape, non_coro_int.shape)

In [ ]:
plt.figure(figsize = (10,14))

plt.subplot(3,2,1)
plt.imshow(master_df)
plt.colorbar(label = 'Counts')
plt.title('Master median dark frame')

plt.subplot(3,2,2)
plt.imshow(np.std(df, axis = 0))
plt.colorbar(label = 'Counts')
plt.title('STD dark frame')

plt.subplot(3,2,3)
plt.imshow(np.log10(np.mean(sci_img, axis = 0)))
plt.colorbar(label = 'Log Counts')
plt.title(f'Raw Mean Image')

plt.subplot(3,2,4)
plt.imshow(np.log10(np.mean(sci_img_processed, axis = 0)*65535))
plt.colorbar(label = f'Log Counts')
plt.title(f'Pre-Processed Mean Image')

cutoff = 20
plt.subplot(3,2,5)
plt.imshow((np.mean(sci_img, axis = 0))[:cutoff, :cutoff])
plt.colorbar(label = ' Counts')
plt.title(f'Raw Mean Image \nBackground [0:{cutoff}, 0:{cutoff}]')

plt.subplot(3,2,6)
plt.imshow((np.mean(sci_img_processed, axis = 0))[:cutoff, :cutoff]*65535)
plt.colorbar(label = f' Counts')
plt.title(f'Pre-Processed Mean Image \nBackground [0:{cutoff}, 0:{cutoff}]')

# Speckle Diversity

## Method Testing

In [ ]:


folder = '/home/aosimul/noah/data/ghost_images/phase_screens_SLM/7_22/'
sci_img_processed = run_pre_processing_with_outliers(img_file = f'{folder}int_0.5ro_10ws_coro', 
                       img_key = 6667, 
                       dark_file = f'{folder}darks',  
                       dark_key= 8475, 
                       ref_psf=f'{folder}psf', 
                       radius = 55,
                       coords= None
                       )

In [ ]:
frames = sci_img_processed

n_frames = frames.shape[0]

# Flatten frames
X = frames.reshape(n_frames, -1).astype(float)

# Mean-center each pixel across the dataset
X -= X.mean(axis=0)



In [ ]:
# PCA (SVD solver)
n_components = 5
pca = PCA(n_components=n_components, svd_solver="full")
pca_coords = pca.fit_transform(X)

#took 2m16s

In [ ]:
# PCA (SVD solver)
n_components = 3
pca2 = PCA(n_components=n_components, svd_solver="full")
pca_coords2 = pca2.fit_transform(X)

#took 2m

In [ ]:
# PCA (SVD randomised solver)
n_components = 5
pca3 = PCA(n_components=n_components, svd_solver="randomized", random_state=42,
)
pca_coords3 = pca3.fit_transform(X)

#took 1.5s

In [ ]:
print(pca_coords)
print(pca_coords2)
print(pca_coords3)

In [ ]:
# Normalise
pca_coords /= np.max(np.abs(pca_coords), axis=0)
pca_coords3 /= np.max(np.abs(pca_coords3), axis=0)





In [ ]:
# Median and MAD along each PCA axis
medians = np.median(pca_coords, axis=0)
mads = np.median(np.abs(pca_coords - medians), axis=0)

medians3 = np.median(pca_coords3, axis=0)
mads3 = np.median(np.abs(pca_coords3 - medians3), axis=0)

In [ ]:

# Avoid division by zero
mads[mads == 0] = 1e-10
mads3[mads3 == 0] = 1e-10
mad_threshold = 5
# Outlier detection
deviation = np.abs(pca_coords - medians)

outlier_matrix = deviation > mad_threshold * mads

bad_mask = np.any(outlier_matrix, axis=1)
good_mask = ~bad_mask

deviation3 = np.abs(pca_coords3 - medians3)

outlier_matrix3 = deviation3 > mad_threshold * mads3

bad_mask3 = np.any(outlier_matrix3, axis=1)
good_mask3 = ~bad_mask3

In [ ]:


# good_mask, bad_mask, outlier_matrix, pca_coords, medians, mads = pca_frame_selection(
#     sci_img_processed,
#     n_components=3,
#     mad_threshold=5,
# )

# #Which PCA rejected each frame
# bad_components = [
#     np.where(outlier_matrix[i])[0]
#     for i in range(len(outlier_matrix))
# ]


fig, ax = plot_pca_frame_selection(
    pca_coords,
    bad_mask,
    medians,
    mads,
    mad_threshold=5,
)

fig, ax = plot_pca_frame_selection(
    pca_coords3,
    bad_mask3,
    medians3,
    mads3,
    mad_threshold=5,
)

In [ ]:
different = bad_mask != bad_mask3
different_idx = np.where(different)[0]

print(f"{different.sum()} frames differ.")

In [ ]:
# Rejected only by PCA
bad_with_full = bad_mask & ~bad_mask3

# Rejected only by old method
bad_with_trunc = bad_mask3 & ~bad_mask

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

cube = sci_img_processed
idx = np.where(different)[0]

ncols = 4
nrows = int(np.ceil(len(idx)/ncols))

fig, axes = plt.subplots(nrows, ncols,
                         figsize=(4*ncols,4*nrows),                         
                         constrained_layout=True)

axes = np.atleast_1d(axes).ravel()

# Use a common normalization for all images
vmin = cube[idx].min()
vmax = cube[idx].max()

im = None
for ax, i in zip(axes, idx):

    im = ax.imshow(
        np.log10(cube[i]),
        origin="lower",
        cmap="gray",
        vmin=np.log10(vmin),
        vmax=np.log10(vmax),
    )

    title = []
    if bad_with_full[i]:
        title.append("bad_with_full only")
    if bad_with_trunc[i]:
        title.append("bad_with_trunc only")

    ax.set_title(f"Frame {i}\n{' '.join(title)}")
    ax.axis("off")

# Remove unused axes
for ax in axes[len(idx):]:
    fig.delaxes(ax)

# One common colorbar
cbar = fig.colorbar(
    im,
    ax=axes[:len(idx)],
    shrink=0.85,
    pad=0.02,
)
cbar.set_label("Pixel value")

plt.suptitle('Compare Full SVD outlier detection to Randomised Truncated SVD')
plt.show()

## Test Pre_processing routine

In [ ]:
folder = '/home/aosimul/noah/data/ghost_images/phase_screens_SLM/7_22/'
sci_img_processed_no_outlier, outlier_matrix = run_pre_processing(img_file = f'{folder}int_0.5ro_10ws_coro', 
                       img_key = 6667, 
                       dark_file = f'{folder}darks',  
                       dark_key= 8475, 
                       ref_psf=f'{folder}psf', 
                       radius = 55,
                       coords= None,
                       n_components=3,
                       mad_threshold=7,
                       PCA_analysis = True
                       )

In [ ]:
folder = '/home/aosimul/noah/data/ghost_images/phase_screens_SLM/7_22/'
sci_img_processed = run_pre_processing_with_outliers(img_file = f'{folder}int_0.5ro_10ws_coro', 
                       img_key = 6667, 
                       dark_file = f'{folder}darks',  
                       dark_key= 8475, 
                       ref_psf=f'{folder}psf', 
                       radius = 55,
                       coords= None
                       )

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm


def plot_outlier_images_by_pc(
    cube,
    outlier_matrix,
    cmap="inferno",
):
    """
    Plot all outlier images grouped by PCA component.

    Parameters
    ----------
    cube : ndarray
        Image cube (n_frames, ny, nx)
    outlier_matrix : ndarray
        Boolean array (n_frames, n_components)
    """

    n_components = outlier_matrix.shape[1]

    # Global normalization over ALL outlier images
    all_indices = np.where(np.any(outlier_matrix, axis=1))[0]

    if len(all_indices) == 0:
        print("No outliers found.")
        return

    data = cube[all_indices]

    # Log scale requires positive values
    vmin = np.percentile(data[data > 0], 1)
    vmax = np.percentile(data[data > 0], 99.5)

    norm = LogNorm(vmin=vmin, vmax=vmax)


    for pc in range(n_components):

        indices = np.where(outlier_matrix[:, pc])[0]

        if len(indices) == 0:
            continue

        ncols = 6
        nrows = int(np.ceil(len(indices) / ncols))

        fig, axes = plt.subplots(
            nrows,
            ncols,
            figsize=(3*ncols, 3*nrows),
        )

        axes = np.atleast_1d(axes).ravel()

        im = None

        for ax, idx in zip(axes, indices):

            # avoid zero values for LogNorm
            image = np.maximum(cube[idx], vmin)

            im = ax.imshow(
                image,
                origin="lower",
                cmap=cmap,
                norm=norm,
            )

            ax.set_title(f"Frame {idx}")
            ax.axis("off")


        # remove unused panels
        for ax in axes[len(indices):]:
            ax.remove()


        # one colorbar for this PC group
        cbar = fig.colorbar(
            im,
            ax=axes[:len(indices)],
            fraction=0.02,
            pad=0.02,
        )

        cbar.set_label("Intensity (log scale)")


        fig.suptitle(
            f"Outliers detected by PC{pc+1} "
            f"({len(indices)} frames)",
            fontsize=16,
        )

        plt.show()

In [ ]:
plot_outlier_images_by_pc(
sci_img_processed, outlier_matrix
)

## Run pre-processing on all images

In [ ]:
save_path = '/home/aosimul/noah/data/ghost_images/10_20ws/'

ori_folder = '/home/aosimul/noah/data/ghost_images/phase_screens_SLM/7_22/'

# Get all files
files = sorted(glob.glob(f"{ori_folder}*.tif"))

# Separate correctly
img_files = [
    f for f in files
    if "coro" in os.path.basename(f)
    and "no_coro" not in os.path.basename(f)
]

psf_files = [
    f for f in files
    if "no_coro" in os.path.basename(f)
]


print(f"Science images: {len(img_files)}")
print(f"PSF images: {len(psf_files)}")


In [ ]:

for img_file in img_files:
    sci_img_processed_no_outlier, outlier_matrix = run_pre_processing(img_file = ori_folder + Path(img_file).stem, 
                        img_key = 6667, 
                        dark_file = f'{ori_folder}darks',  
                        dark_key= 8475, 
                        ref_psf=f'{ori_folder}psf', 
                        radius = 55,
                        n_components=3,
                        mad_threshold=7,
                        PCA_analysis = True
                        )
    name = os.path.basename(img_file)          # int_1.0ro_10ws_no_coro_2026-07-22T11-24-21.543_0
    save_name = name.split("_coro")[0]  # int_1.0ro_10ws_no_coro
    
    sci_img_processed = run_pre_processing_with_outliers(img_file = ori_folder + Path(img_file).stem,
                       img_key = 6667, 
                       dark_file = f'{ori_folder}darks',  
                       dark_key= 8475, 
                       ref_psf=f'{ori_folder}psf', 
                       radius = 55,
                       coords= None
                       )
    plot_outlier_images_by_pc(
        sci_img_processed, outlier_matrix
        )
    np.save(f'{save_path}{save_name}.npy', sci_img_processed_no_outlier)

for psf_file in psf_files:
    psf_img_processed = run_pre_processing_with_outliers(img_file = ori_folder + Path(psf_file).stem, 
                        img_key = 8475, 
                        dark_file = f'{ori_folder}darks',  
                        dark_key= 8475, 
                        ref_psf=f'{ori_folder}psf', 
                        radius = 55
                        )
    psf_med = np.sum(psf_img_processed, axis = 0)
    name = os.path.basename(psf_file)          # int_1.0ro_10ws_no_coro_2026-07-22T11-24-21.543_0
    save_name = name.split("_no")[0]  # int_1.0ro_10ws_no_coro
    np.save(f'{save_path}psf_{save_name}.npy', psf_med)

In [ ]:
name = os.path.basename(img_file)          # int_1.0ro_10ws_no_coro_2026-07-22T11-24-21.543_0
save_name = name.split("_2026")[0]  # int_1.0ro_10ws_no_coro
print(f'{save_name}.png')